In [1]:
%matplotlib inline

import os, json
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal as sp_signal
from scipy.stats import skew, kurtosis
from scipy.ndimage import gaussian_filter1d
from collections import Counter

np.random.seed(42)
os.makedirs("outputs", exist_ok=True)

FEAT_NAMES = [
    'ApEn','SampEn','Shannon','Spectral','PermEn',
    'Skewness','Kurtosis','Higuchi FD','SNR (dB)'
]

print("Step 2 Environment Ready")

Step 2 Environment Ready


In [2]:
import os
print("Current folder:", os.getcwd())
print("Files here:", os.listdir())

Current folder: E:\ppg_research
Files here: ['.ipynb_checkpoints', '201902020222_Data.csv', 'outputs', 'reconstruction_metrics.csv', 'run_all.py', 'step1_preprocess.ipynb', 'step1_preprocessing.py', 'step2_SQA.ipynb', 'step2_sqa.py', 'step3_autoencoder.py', 'step4_quality_verification.py', 'step5_visualization.py', 'Untitled.ipynb']


In [3]:
import os
print(os.listdir("outputs"))

['clean_segs.npy', 'fig1_clean_noisy_reconstructed.png', 'fig3_metrics_improvement.png', 'mu_c.npy', 'ppg_raw.npy', 'ppg_smooth.npy', 'reconstruction_metrics.csv', 'signal_reconstruction_comparison.png', 'std_c.npy', 'step1a_raw_vs_filtered.png', 'step3a_training_loss.png', 'step3b_clean_noisy_reconstructed.png', 'time_domain_error_comparison.png', 'Xc.npy', 'Xn.npy']


In [4]:
print("\nChecking Step 1 outputs...")

required = ["Xc.npy", "Xn.npy", "clean_segs.npy"]

for f in required:
    if not os.path.exists("outputs/" + f):
        raise FileNotFoundError(f"Missing {f}. Run Step 1 first.")

print("All required files found.")


Checking Step 1 outputs...
All required files found.


In [5]:
print("\n[1/4] Loading preprocessed data...")

Xc = np.load("outputs/Xc.npy")
Xn = np.load("outputs/Xn.npy")
clean_segs = np.load("outputs/clean_segs.npy")

print("Xc shape :", Xc.shape)
print("Xn shape :", Xn.shape)
print("Segments :", len(clean_segs))


[1/4] Loading preprocessed data...
Xc shape : (1707, 200)
Xn shape : (1707, 200)
Segments : 569


In [6]:
def approx_entropy(sig, m=2, r=0.2):

    r *= np.std(sig)+1e-10
    N = len(sig)

    def phi(m_):
        xm = np.array([sig[i:i+m_] for i in range(N-m_)])
        C = np.sum(np.max(np.abs(xm[:,None]-xm[None,:]),2)<=r,1)/(N-m_)
        return np.mean(np.log(C+1e-10))

    return abs(phi(m)-phi(m+1))


def samp_entropy(sig, m=2, r=0.2):

    r *= np.std(sig)+1e-10
    N = len(sig)

    def cnt(m_):
        xm = np.array([sig[i:i+m_] for i in range(N-m_)])
        return np.sum(np.max(np.abs(xm[:,None]-xm[None,:]),2)<=r)-(N-m_)

    return -np.log((cnt(m+1)+1e-10)/(cnt(m)+1e-10))

In [7]:
def shannon_ent(sig, bins=32):

    h,_ = np.histogram(sig, bins=bins, density=True)
    h = h[h>0]

    return -np.sum(h*np.log2(h+1e-10))


def spectral_ent(sig, fs=100):

    _,psd = sp_signal.welch(sig, fs=fs, nperseg=min(64,len(sig)))

    p = psd/(np.sum(psd)+1e-10)
    p = p[p>0]

    return -np.sum(p*np.log2(p+1e-10))


def perm_ent(sig, m=3, tau=1):

    N=len(sig)

    pats = [
        tuple(np.argsort([sig[i+j*tau] for j in range(m)]))
        for i in range(N-(m-1)*tau)
    ]

    cnt = Counter(pats)
    tot = sum(cnt.values())

    prb = np.array([v/tot for v in cnt.values()])

    return -np.sum(prb*np.log2(prb+1e-10))


def higuchi_fd(sig, k_max=8):

    N=len(sig)
    L=[]

    for k in range(1,k_max+1):

        Lk=[]

        for m in range(1,k+1):

            idx=list(range(m-1,N,k))

            if len(idx)<2:
                continue

            Lk.append(
                np.sum(np.abs(np.diff(sig[idx])))*
                (N-1)/(len(idx)*k)
            )

        if Lk:
            L.append(np.mean(Lk))

    if len(L)<2:
        return 1.0

    return np.polyfit(
        np.log(np.arange(1,len(L)+1)),
        np.log(np.array(L)+1e-10),
        1
    )[0]

In [8]:
def seg_snr(seg):

    ref = gaussian_filter1d(seg,2)

    return 10*np.log10(
        np.var(seg)/(np.var(seg-ref)+1e-10)
    )


def compute_features(seg):

    return np.array([
        approx_entropy(seg),
        samp_entropy(seg),
        shannon_ent(seg),
        spectral_ent(seg),
        perm_ent(seg),
        skew(seg),
        kurtosis(seg),
        higuchi_fd(seg),
        seg_snr(seg)
    ])

In [9]:
def sqa_grade(feat):

    snr, apen, kurt = feat[8], feat[0], feat[6]

    if snr >= 10 and apen < 0.6:
        return 'clean'

    elif snr >= 5 or (apen < 0.8 and abs(kurt) < 5):
        return 'partly_clean'

    else:
        return 'corrupted'

In [10]:
N_SQA = min(len(clean_segs), 80)

idx = np.random.choice(len(clean_segs), N_SQA, replace=False)

gc, gn = [], []
feats_c, feats_n = [], []


for i in idx:

    fc = compute_features(clean_segs[i])

    noise = np.random.normal(0,0.25*np.std(clean_segs[i]),len(clean_segs[i]))
    fn = compute_features(clean_segs[i] + noise)

    feats_c.append(fc)
    feats_n.append(fn)

    gc.append(sqa_grade(fc))
    gn.append(sqa_grade(fn))

feats_c = np.array(feats_c)
feats_n = np.array(feats_n)

print("SQA feature extraction and grading completed.")

SQA feature extraction and grading completed.
